In [16]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.9 MB 640.0 kB/s eta 0:00:14
   ---------------------------------------- 0.1/8.9 MB 465.5 kB/s eta 0:00:19
    --------------------------------------- 0.1/8.9 MB 652.2 kB/s eta 0:00:14
    --------------------------------------- 0.2/8.9 MB 748.1 kB/s eta 0:00:12
   - -------------------------------------- 0.2/8.9 MB 808.4 kB/s eta 0:00:11
   - -------------------------------------- 0.3/8.9 MB 930.9 kB/s eta 0:00:10
   - -------------------------------------- 0.3/8.9 MB 893.0 kB/s eta 0:00:10
   - -------------------------------------- 0.4/8.9 MB 930.9 kB/s eta 0:00:10
   - -------------------------------------- 0.4/8.9 MB 922.1 kB/s eta 0:00:10
   -- ------------------------------------- 0.5/8.9 MB 962.6 kB/s eta 0:00:09
   -- ------------------------------------- 0.6/8.9 MB 1.0 MB/s eta 0:00:09
   -- 


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')

In [3]:
df_train

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.300
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.700
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.000
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.900
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,629995,18,female,b.tech,4.86,70.7,yes,4.1,good,mixed,high,moderate,69.500
629996,629996,21,female,ba,7.08,54.4,yes,4.5,average,mixed,low,moderate,78.900
629997,629997,24,male,bca,0.64,44.2,yes,4.3,poor,online videos,low,moderate,19.599
629998,629998,20,male,b.com,1.54,75.1,yes,8.2,average,group study,high,moderate,59.100


In [4]:
df_test

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate
...,...,...,...,...,...,...,...,...,...,...,...,...
269995,899995,21,other,b.com,2.55,82.3,yes,8.4,average,mixed,medium,hard
269996,899996,17,female,b.com,0.49,46.4,yes,8.8,good,mixed,low,easy
269997,899997,22,male,bba,6.62,74.7,yes,5.5,good,coaching,high,easy
269998,899998,22,other,ba,4.08,51.8,yes,8.7,poor,online videos,high,moderate


In [5]:
class_features = [i for i in df_train.columns if df_train[i].dtype == 'object']

for col in class_features:
    df_train[col] = df_train[col].astype('category')
    df_test[col] = df_test[col].astype('category')

for col in class_features:
    print(pd.unique(df_train[col]))

['female', 'other', 'male']
Categories (3, object): ['female', 'male', 'other']
['b.sc', 'diploma', 'bca', 'b.com', 'ba', 'bba', 'b.tech']
Categories (7, object): ['b.com', 'b.sc', 'b.tech', 'ba', 'bba', 'bca', 'diploma']
['no', 'yes']
Categories (2, object): ['no', 'yes']
['average', 'poor', 'good']
Categories (3, object): ['average', 'good', 'poor']
['online videos', 'self-study', 'coaching', 'group study', 'mixed']
Categories (5, object): ['coaching', 'group study', 'mixed', 'online videos', 'self-study']
['low', 'medium', 'high']
Categories (3, object): ['high', 'low', 'medium']
['easy', 'moderate', 'hard']
Categories (3, object): ['easy', 'hard', 'moderate']


In [6]:
test_id = df_test['id']

df_train.drop(columns = 'id', inplace = True)
df_test.drop(columns = 'id', inplace = True)

In [7]:
X = df_train.drop(columns = 'exam_score')
y = df_train['exam_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [8]:
params = {'random_seed': 42,
          'verbose': 100,
          'cat_features': class_features}

cat_model = CatBoostRegressor(**params)
cat_model.fit(X_train, y_train)

y_pred = cat_model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
#pred = cat_model.predict(test)
rmse

Learning rate set to 0.109437
0:	learn: 17.5977570	total: 735ms	remaining: 12m 14s
100:	learn: 8.8421599	total: 51.1s	remaining: 7m 34s
200:	learn: 8.8083692	total: 1m 44s	remaining: 6m 57s
300:	learn: 8.7863857	total: 2m 35s	remaining: 6m 1s
400:	learn: 8.7693139	total: 3m 24s	remaining: 5m 5s
500:	learn: 8.7554787	total: 4m 12s	remaining: 4m 11s
600:	learn: 8.7436542	total: 4m 59s	remaining: 3m 18s
700:	learn: 8.7324395	total: 5m 48s	remaining: 2m 28s
800:	learn: 8.7219904	total: 6m 37s	remaining: 1m 38s
900:	learn: 8.7113826	total: 7m 28s	remaining: 49.2s
999:	learn: 8.7021155	total: 8m 17s	remaining: 0us


8.748790641273057

In [9]:
pred = cat_model.predict(df_test)

In [10]:
output = pd.DataFrame({'id': test_id, 'exam_score': pred})
output.to_csv('data/cat_model.csv', index=False)

# kaggle - Score: 8.73312